# Suite TXN — Transactions: mutation expressions over frames, and what they are built from

A transaction is a mutation expression over a stream of frames (docs/STREAMING_QUERIES.md; docs/OBJECT_IDENTITY.md,
Decided, 85 to 97): its terms are reference objects of the `Mutations` schemas, linked by relations, each term's
pattern a `Schemas.OfPattern` built through its link, and a term chosen through a kind union, `Mutations.OfTerm`. A
pattern's `spec` says what must hold of a frame, written in the builders' notation over the frame's symbols and recorded
as mutations. A keyed adjacency takes a mapping in place of the peer shorthand (FRAMEWORK.md, Resolved).

Python and TypeScript run this suite with the same case IDs. The case setup is below.

In [ ]:
import gc

from mbse.Schemas.Framework import Builders as B, Paths, Plain, Proxies, Schemas as S, Transactions
from mbse.Schemas.Framework.Errors import ReadOnlyError
from support import raises


def crm(store):
    """crm.Person, who knows persons since a year; crm.Book, a singleton owning its contacts by name; crm.Note, counted."""
    Int, Str = store.Schemas.OfInt, store.Schemas.OfStr
    Knows = store.Schemas.OfRelation(lambda r: r.name("crm.Knows").links([lambda t: t.name("knower"), lambda t: t.name("known")])
                                     .properties({"since": Int}))
    Person = store.Schemas.OfObject(lambda o: o.name("crm.Person").ref(True).properties({"name": Str, "age": Int}))
    Holds = store.Schemas.OfRelation(lambda r: (
        r.name("crm.Holds").links([lambda t: t.name("book"), lambda t: t.name("contact")]).properties({"key": Str})
        .uniques([["book", "key"], ["contact"]])))
    store.Schemas.OfObject(Person, lambda o: o.adjacencies([
        lambda t: t.name("knows").relation([Knows]).me("knower"),
        lambda t: t.relation([Knows]).me("known"),
        lambda t: t.name("book").relation([Holds]).me("contact").owner(True),
    ]))
    Book = store.Schemas.OfObject(lambda o: o.name("crm.Book").ref(True).singleton("crm.TheBook").adjacencies({
        "contacts": lambda t: t.relation([Holds]).me("book")}))
    Note = store.Schemas.OfObject(lambda o: o.name("crm.Note").ref(True).properties({"text": Str}))
    Meets = store.Schemas.OfRelation(lambda r: r.name("crm.Meets").links([lambda t, n=n: t.name(n) for n in ("a", "b", "c")]))
    Kind = store.Schemas.OfUnion(lambda u: u.name("crm.Kind").branches([lambda t: t.name("person").schema([Person]),
                                                                         lambda t: t.name("note").schema([Note])]))
    Picks = store.Schemas.OfRelation(lambda r: r.name("crm.Picks").links([lambda t: t.name("owner"),
                                                                       lambda t: t.name("pick").schema([Kind])])
                                     .properties({"rank": Int}))
    Pins = store.Schemas.OfRelation(lambda r: r.name("crm.Pins").links([lambda t: t.name("note"),
                                                                     lambda t: t.name("book").schema([Book])])
                                    .properties({"at": Int}))
    Trio = store.Schemas.OfRelation(lambda r: r.links([lambda t, n=n: t.name(n) for n in ("a", "b", "c")]))  # unnamed
    Odd = store.Schemas.OfObject(lambda o: o.name("crm.Odd").ref(True).adjacencies([lambda t: t.name("x").me("a")]))  # no relation yet
    Odds = store.Schemas.OfRelation(lambda r: r.name("crm.Odds").links([lambda t: t.name("note"), lambda t: t.name("odd").schema([Odd])])
                                    .properties({"k": Int}))
    store.Schemas.OfObject(Person, lambda o: o.adjacencies({"meets": lambda t: t.relation([Meets]).me("a"),
                                                            "trio": lambda t: t.relation([Trio]).me("a")}))
    store.Schemas.OfObject(Note, lambda o: o.adjacencies({"picks": lambda t: t.relation([Picks]).me("owner"),
                                                          "pins": lambda t: t.relation([Pins]).me("note"),
                                                          "odds": lambda t: t.relation([Odds]).me("note")}))
    return Person, Knows, Book, Note


store = B.Store()
Int, Str = store.Schemas.OfInt.as_schema(), store.Schemas.OfStr.as_schema()
Person, Knows, Book, Note = crm(store)

## TXN-01 · A kind union: a link's schema a union of reference object schemas, each branch a kind built through it

In [ ]:
Mutations = store.Mutations
OfTerm = Mutations.OfTerm.as_schema()
assert S._kinds(OfTerm) == {"map": Mutations.OfMap.as_schema(), "reduce": Mutations.OfReduce.as_schema(),
                            "partition": Mutations.OfPartition.as_schema(), "update": Mutations.OfUpdate.as_schema(),
                            "upsert": Mutations.OfUpsert.as_schema()}
assert S._kinds(Person) is None and S._kinds(store.Schemas.OfStr.as_schema()) is None
assert all(store.registered(name).validate() == [] for name in store.names() if name.startswith("Mutations."))
mixed = store.Schemas.OfUnion(lambda u: u.branches([lambda t: t.name("p").schema([Person]),
                                                    lambda t: t.name("v").schema([store.Schemas.OfObject(lambda o: o)])]))
assert store.Schemas.OfRelation(lambda r: r.links([lambda t: t.name("a"), lambda t: t.name("b").schema([mixed])])).validate() == [
    "link 'b': a kind union's branches are reference object schemas"]
# Building through the link: a branch, called, builds an object of its kind; the peer shorthand's visitor and an entry
# builder each have the branches as members.
tx = store.Mutations.OfTransaction(lambda t: t.uid("t-1").mutations([lambda mu: mu.upsert(lambda u: (
    u.key("k").then([lambda w: w.update(lambda d: d)])))]))
root = tx.mutations[0].term
assert root.as_schema() is Mutations.OfUpsert.as_schema() and str(root.key) == "k"
assert root.then[0].next.as_schema() is Mutations.OfUpdate.as_schema()
other = store.Mutations.OfTransaction(lambda t: t.uid("t-2").mutations(lambda a: a.insert(lambda e: e.map(lambda m: m))))
assert other.mutations[0].term.as_schema() is Mutations.OfMap.as_schema()
kept = store.Mutations.OfTransaction(lambda t: t.uid("t-3").mutations(lambda a: a.insert(lambda e: e.update(root.then[0].next))))
assert kept.mutations[0].term is root.then[0].next  # a kind given an object links it
with raises(TypeError, match="link 'term' is a kind union: build through one of its kinds, map, reduce, partition, update, upsert"):
    store.Mutations.OfTransaction(lambda t: t.mutations([lambda mu: mu.uid]))  # a member of no kind: which object?
with raises(TypeError, match="link 'term' is a kind union: build through one of its kinds"):
    store.Mutations.OfTransaction(lambda t: t.mutations(lambda a: a.insert(lambda e: e.term(lambda x: x))))
seen = []
store.Mutations.OfTransaction(lambda t: t.mutations(lambda a: a.insert(lambda e: seen.append(repr(e.upsert)) or e.upsert(lambda u: u))))
assert seen == ["<kind of link 'term'>"]

## TXN-02 · Mappings for keyed adjacencies: each item an entry, its key the one property, its value a callback or given to the one member that takes it

In [ ]:
assert list(Person.properties) == ["name", "age"] and Person.properties["age"].schema is Int  # the schema builders'
Tagged = store.Schemas.OfObject(lambda o: o.properties({"tag": lambda t: t.schema([Str]).description("a label")}))
assert Tagged.properties["tag"].description == "a label"
with raises(TypeError, match="'schemas' takes a mapping only where its relation has two links and one property"):
    store.Schemas.OfAny(lambda a: a.schemas({"n": Int}))
with raises(TypeError, match="more than one member of the entry takes 'text': give a callback"):
    store.Schemas.OfObject(lambda o: o.properties({"x": "text"}))  # its description and its naming pattern
with raises(TypeError, match="no member of the entry takes 3: give a callback"):
    store.Schemas.OfObject(lambda o: o.properties({"x": 3}))
root_module = store.module()
store.Schemas.OfModule(root_module, lambda m: m.schemas({"Tagged": Tagged}))  # the link alone takes a schema
assert root_module.schemas["Tagged"] is Tagged
book = store.singleton("crm.TheBook")
store.crm.Book(book, lambda b: b.contacts({"ada": lambda e: e.name("Ada"), "bob": lambda e: e.name("Bob").age(36)}))
ada, bob = book.contacts({"key": "ada"}), book.contacts({"key": "bob"})  # the proxies': each a contact, by its key
assert [(str(e.key), str(e.contact.name)) for e in book.contacts] == [("ada", "Ada"), ("bob", "Bob")] and int(bob.age) == 36
store.crm.Person(ada, lambda c: c.knows({2020: lambda e: e.known(bob)}))
assert [(int(e.since), e.known) for e in ada.knows] == [(2020, bob)]
with raises(TypeError, match="more than one member of the entry takes <crm.Person>: give a callback"):
    store.crm.Person(ada, lambda c: c.knows({2021: bob}))  # the link, and the linked person's own 'knows'
with raises(TypeError, match="no member of the entry takes 1.5: give a callback"):
    store.crm.Book(book, lambda b: b.contacts({"x": 1.5}))
with raises(TypeError, match="adjacency 'mutations' takes a mapping only where its relation has two links and one property"):
    store.Mutations.OfTransaction(lambda t: t.mutations({"x": lambda e: e}))
with raises(TypeError, match="no member of the entry takes 'x': give a callback"):
    store.Schemas.OfModule(root_module, lambda m: m.schemas({"x": "x"}))
with raises(TypeError, match="'schemas' takes an iterable of what it links, or of entries, a mapping, or a callback"):
    store.Schemas.OfAny(lambda a: a.schemas("x"))
# A link of a kind union, or of a schema it declares, takes an object of it; a callback chooses a kind.
note = store.crm.Note(lambda n: n.picks({1: ada, 2: lambda e: e.note(lambda m: m.text("aside"))}).pins({3: book}))
assert [(int(e.rank), e.pick.as_schema().name) for e in note.picks] == [(1, "crm.Person"), (2, "crm.Note")]
assert [(int(e.at), e.book) for e in note.pins] == [(3, book)]
with raises(TypeError, match="no member of the entry takes 'zz': give a callback"):
    store.crm.Note(lambda n: n.odds({1: "zz"}))  # an adjacency with no relation yet takes nothing
outside = B.OfObject.Builder().properties({"n": lambda t: t.description("d")}).create()  # outside a store too
assert outside.properties["n"].description == "d"

## TXN-03 · A transaction's expression: reference objects linked by relations, each term's pattern a schema object built through its link and held while its term is

In [ ]:
def upserting():
    return store.Mutations.OfTransaction(lambda t: t.uid("t-9").mutations([lambda mu: mu.upsert(lambda u: u.exists([
        lambda p: p.symbols({"ada": Person}).description("whom")]))]))


held = len(store._unnamed)
tx = upserting()
pattern = tx.mutations[0].term.exists[0].pattern
assert isinstance(pattern, S.OfPattern.Data) and list(pattern.symbols) == ["ada"] and pattern.description == "whom"
assert store.holds(pattern) and store.holds(pattern.symbols["ada"]) and len(store._unnamed) == held + 2
with raises(LookupError, match="a pattern with no name of its own"):
    Paths.of(pattern)
del tx, pattern
gc.collect()
assert len(store._unnamed) == held  # a term's pattern goes with its transaction (Decided, 94)
Edit = store.Schemas.OfPattern(lambda p: p.name("crm.Edit").symbols({"ada": Person}))
store.Mutations.OfUpdate(lambda u: u.apply([lambda p: p.pattern(Edit).description("edits")]))  # linked, then updated
assert Edit.description == "edits"
assert store._meta_builder("crm.Person") is None and store._meta_builder("Schemas.Nope") is None
assert repr(store._meta_builder("Schemas.OfPattern")) == "<builder of a pattern>"
assert store._meta_builder_class(Edit) is B.PatternBuilder

## TXN-04 · A spec: written in the builders' notation over the frame's symbols, recorded as mutations, by symbol

In [ ]:
def ref(name):
    return {"ref": {"symbol": name}}


def path_of(target):
    return {"ref": {"path": Plain.ToPlain(store)(store.Paths.OfRoot.as_schema(), Paths.of(target))}}


def link(name, target):
    return {"as_link": {"name": name, "mutations": [{"set": target}]}}


def prop(name, value):
    return {"as_property": {"name": name, "mutations": [{"set": {"value": value}}]}}


def entries(name, *items):
    return {"as_adjacency": {"name": name, "mutations": [{"insert": {"entries": [{"mutations": list(i)} for i in items]}}]}}


Meet = store.Schemas.OfPattern(lambda p: p.symbols({"ada": Person, "bob": Person}).spec([
    lambda _: _.bob.name("Bob").knows([_.ada]),
    lambda _: _.ada.name("Ada").age(None).knows([lambda e: e.known(_.bob).since(2020)]),
]))
assert Meet.spec == (
    {"bob": [prop("name", "Bob"), entries("knows", [link("known", ref("ada"))])]},
    {"ada": [prop("name", "Ada"), {"as_property": {"name": "age", "mutations": [{"clear": {}}]}},
             entries("knows", [link("known", ref("bob")), prop("since", 2020)])]},
)
book = store.singleton("crm.TheBook")
Owned = store.Schemas.OfPattern(lambda p: p.symbols({"bob": Person}).spec(  # one callback; a path; a mapping
    lambda _: _.bob.book({"bob": book}).knows(lambda a: a.insert(lambda e: e.known(lambda n: n.name("Dan")).since(1)))))
assert Owned.spec == ({"bob": [
    entries("book", [link("book", path_of(book)), prop("key", "bob")]),
    entries("knows", [link("known", {"def": {"schema": "crm.Person", "mutations": [prop("name", "Dan")]}}), prop("since", 1)]),
]},)
Defines = store.Schemas.OfPattern(lambda p: p.symbols({"P": store.Schemas.OfObject}).spec([
    lambda _: _.P.name("crm.Pet").ref(True).properties({"name": Str, "legs": lambda t: t.schema([Int])}),
]))
property_schema = lambda schema: {"def": {"schema": "Schemas.OfProperty", "mutations": [
    entries("schema", [link("schema", path_of(schema))])]}}
assert Defines.spec == ({"P": [prop("name", "crm.Pet"), prop("ref", True), entries(
    "properties", [link("property", property_schema(Str)), prop("name", "name")],
    [link("property", property_schema(Int)), prop("name", "legs")])]},)
Terms = store.Schemas.OfPattern(lambda p: p.symbols({"t": store.Mutations.OfTransaction}).spec([
    lambda _: _.t.uid("u").mutations([lambda e: e.upsert(lambda u: u.key("k"))]),  # a kind union: a kind defined
    lambda _: _.t.mutations([lambda e: e.term(_.t)]),
]))
assert Terms.spec[0] == {"t": [prop("uid", "u"), entries("mutations", [link("term", {"def": {
    "schema": "Mutations.OfUpsert", "mutations": [prop("key", "k")]}})])]}
assert Terms.spec[1] == {"t": [entries("mutations", [link("term", ref("t"))])]}
# Reading through the frame, and what a spec refuses.
seen = []
store.Schemas.OfPattern(lambda p: p.symbols({"ada": Person}).spec([
    lambda _: seen.extend([repr(_), repr(_.ada), _.ada.name.get(), _.ada.as_schema() is Person,
                           _.nope.as_schema(), _.ada.nope.as_schema(), _.ada.knows.as_schema().name,
                           hasattr(_, "_hidden"), hasattr(_.ada, "_hidden"), hasattr(_.ada.name, "_hidden")]),
    lambda _: _.ada.knows(lambda a: a.insert(lambda e: seen.extend([
        e.as_schema() is Knows.entry, hasattr(e, "_hidden"), e.nope.as_schema()]) or e.known(_.ada))),
    lambda _: _.ada.knows([lambda e: e.known(lambda n: seen.append(repr(n)))]),
]))
assert seen == ["<frame>", "<spec of ada>", None, True, None, None, "knows", False, False, False, True, False, None,
                "<spec of an object>"]
Given = store.Schemas.OfPattern(lambda p: p.symbols({"ada": Person, "bob": Person}).spec([
    lambda _: _.bob.knows({2020: "Dan"}).book({"b": _.ada}),  # a property of the linked object; an adjacency of it
    lambda _: _.bob.knows([lambda e: e.known(lambda n: n.name("Eve")).age(3)]),  # the object it defines, written on
]))
assert Given.spec[0] == {"bob": [
    entries("knows", [link("known", {"def": {"schema": "crm.Person", "mutations": [prop("name", "Dan")]}}), prop("since", 2020)]),
    entries("book", [link("book", {"def": {"schema": "crm.Book", "mutations": [entries("contacts", [link("contact", ref("ada"))])]}}),
                     prop("key", "b")])]}
assert Given.spec[1] == {"bob": [entries("knows", [link("known", {"def": {"schema": "crm.Person", "mutations": [
    prop("name", "Eve"), prop("age", 3)]}})])]}
note = store.crm.Note(lambda n: n)
for spec, error, says in (
    (lambda _: _.x.name("x"), TypeError, "symbol 'x' has no schema: a spec writes what its object's schema declares"),
    (lambda _: _.ada.knows(3), TypeError, "adjacency 'knows' takes an iterable of entries or of what it links, a mapping"),
    (lambda _: _.ada.book({"a": 1.5}), TypeError, "no member of the entry takes 1.5: give a callback"),
    (lambda _: _.ada.knows([lambda e: e.known(_.ada).name("Z")]), ValueError,
     "link 'known' links an object a spec does not define: write its members in its own spec"),
    (lambda _: _.ada.knows([lambda e: e.known(note)]), ValueError,
     "a counted 'crm.Note' has no path: a spec refers to it through a symbol"),
    (lambda _: _.ada.meets([lambda e: e]), TypeError,
     "adjacency 'meets': entries of crm.Meets have more than two links; give each entry's links"),
    (lambda _: _.ada.trio([lambda e: e]), TypeError, "adjacency 'trio': entries of its relation have more than two links"),
    (lambda _: _.ada.knows({2020: _.ada}), TypeError, "more than one member of the entry takes <spec of ada>: give a callback"),
    (lambda _: _.ada.meets({1: _.ada}), TypeError,
     "adjacency 'meets' takes a mapping only where its relation has two links and one property"),
    (lambda _: _.ada.knows([lambda e: e.known(lambda n: n.knows([lambda f: f.known(n)]))]), ValueError,
     "an object a spec defines is linked where it is defined"),
    (lambda _: _.t.mutations([lambda e: e.term(lambda x: x)]), TypeError,
     "link 'term' is a kind union: define through one of its kinds, map, reduce, partition, update, upsert"),
    (lambda _: _.t.mutations([lambda e: e.uid("u")]), TypeError,
     "link 'term' is a kind union: define through one of its kinds, map, reduce, partition, update, upsert"),
):
    with raises(error, match=says):
        store.Schemas.OfPattern(lambda p: p.symbols({"ada": Person, "x": lambda t: t,
                                                     "t": store.Mutations.OfTransaction}).spec([spec]))
entry = []
store.Schemas.OfPattern(lambda p: p.symbols({"ada": Person}).spec([lambda _: _.ada.knows([lambda e: entry.append(e) or e.known(_.ada)])]))
with raises(AttributeError):
    entry[0].since = 1  # a member is written by calling it
with raises(TypeError, match="a spec is written in a store, whose schemas its symbols name"):
    B.PatternBuilder().spec([lambda _: _])
with raises(ReadOnlyError):
    B.PatternBuilder(Meet, False, store).spec([lambda _: _])